.option("delta.appendOnly", "true")
Delta enforces append-only semantics, meaning:

✅ INSERT → allowed
❌ UPDATE → blocked
❌ DELETE → blocked
❌ MERGE with updates → blocked

In [0]:
%sql
-- Create an append-only table
CREATE OR REPLACE TABLE data_engineering_workshop.antony.events (
  id INT,
  event_time TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.appendOnly' = 'true');

-- This will succeed
INSERT INTO data_engineering_workshop.antony.events VALUES (1, current_timestamp());

-- This will fail with an AnalysisException
DELETE FROM data_engineering_workshop.antony.events WHERE id = 1;


In [0]:
%skip
THis is just an information cell to check %skip

In [0]:
# Create DataFrame
data = [("event1",), ("event2",)]
df = spark.createDataFrame(data, ["name"])

# Write to table, enforced to append only
df.write.format("delta") \
  .option("delta.appendOnly", "true") \
  .mode("append") \
  .saveAsTable("data_engineering_workshop.antony.events_table")


In [0]:

spark.sql("""
UPDATE data_engineering_workshop.antony.events_table
SET name = 'event_updated'
WHERE name = 'event1'
""")

AnalysisException: 
This table is configured to only allow appends. 
Updates and deletes are not allowed.

In [0]:
spark.sql("""
DELETE FROM data_engineering_workshop.antony.events_table
WHERE name = 'event2'
""")

AnalysisException: This table is configured to only allow appends. Updates and deletes are not allowed.

## DeltaTable.forName() vs DeltaTable.forPath()
Both are used to get a reference to an existing Delta table, but they differ in how you identify the table

1️⃣ DeltaTable.forName()
👉 Used when the table is registered in the Metastore (Unity Catalog or Hive Metastore)
_**You refer to the table using: catalog.schema.table**_

_** ✅ When to use forName()?**_

✔ Table created using:
CREATE TABLE catalog.schema.table ...

✔ Managed tables
✔ External tables registered in metastore
✔ Unity Catalog environment (recommended way)

🔹 2️⃣ DeltaTable.forPath()
👉 Used when you know the physical storage path

You directly point to the Delta folder location.
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "abfss://container@storageaccount.dfs.core.windows.net/events/"
)

🧠 What happens here?

Spark:

Goes directly to that folder
Looks for _delta_log
Treats it as a Delta table
No metastore lookup involved.

✅ When to use forPath()?

✔ Raw Delta folder
✔ Table NOT registered in metastore
✔ Temporary pipelines
✔ Migration scenarios
✔ Debugging broken metastore

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "data_engineering_workshop.antony.events_table")

delta_table.update(
  condition = "name = 'event1'",
  set = { "name": "'event_updated'" }
)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "data_engineering_workshop.antony.events_table")

delta_table.delete("name = 'event1'")

🎯 **Simple Mental Model**
_Scenario_	                    _Use_
Table visible in SHOW TABLES	  forName()
You only have storage path	    forPath()
Unity Catalog production setup	forName()
Raw ADLS/S3 location	          forPath()

🔍** Deep Technical Difference**
**spark.table()**
Uses Catalog API
Returns logical plan
No format validation until action
**DeltaTable.forName()**
Loads DeltaTable object
Validates provider
Loads Delta log
Enables transactional operations

**Interview-Ready Explanation**

_spark.table()_ returns a DataFrame and works for any table type including views and non-Delta tables.
_DeltaTable.forName()_ specifically validates that the table is a Delta table and loads the transaction log, so it fails if the table is not Delta, is a view, or has permission issues.